## Normalized AlexNet derived FID computation for CIFAR-10

This notebook can be used to replicate the results reported in  "Generative adversarial learning can explain why imagination seems less real as we grow up" by Ozsu, Petrova, Dekker & Dijkstra.

Scripts would pass 10000 generated and 50000 real images (once) through AlexNet and calculate FID per epoch. It also calculates a real-real baseline to normalize the FID numbers using the inherent ceiling for each layer. To account for the dimensionality differences and ill-defined FID in low-dimensional spaces SPP and PCA are applied per convolutional layer as described in the paper.

To run this scripts, you would need to have the weights available which can be gathered from the OSF files. In this implementation, the weights are imported via Google Drive but other methods are also possible.

For any questions or if you detect a bug: a.ozsu@ucl.ac.uk


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!unzip "/content/drive/MyDrive/checkpoints_newarch.zip" -d "/content"

Mounted at /content/drive
Archive:  /content/drive/MyDrive/checkpoints_newarch.zip
   creating: /content/checkpoints/
  inflating: /content/checkpoints/C_epoch_1.pth  
  inflating: /content/checkpoints/C_epoch_10.pth  
  inflating: /content/checkpoints/C_epoch_100.pth  
  inflating: /content/checkpoints/C_epoch_11.pth  
  inflating: /content/checkpoints/C_epoch_12.pth  
  inflating: /content/checkpoints/C_epoch_13.pth  
  inflating: /content/checkpoints/C_epoch_14.pth  
  inflating: /content/checkpoints/C_epoch_15.pth  
  inflating: /content/checkpoints/C_epoch_16.pth  
  inflating: /content/checkpoints/C_epoch_17.pth  
  inflating: /content/checkpoints/C_epoch_18.pth  
  inflating: /content/checkpoints/C_epoch_19.pth  
  inflating: /content/checkpoints/C_epoch_2.pth  
  inflating: /content/checkpoints/C_epoch_20.pth  
  inflating: /content/checkpoints/C_epoch_21.pth  
  inflating: /content/checkpoints/C_epoch_22.pth  
  inflating: /content/checkpoints/C_epoch_23.pth  
  inflating: /co

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
from scipy import linalg
from sklearn.covariance import LedoitWolf
from tqdm import tqdm
import gc
import matplotlib.pyplot as plt

# Device setup
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# =============================================================================
# CONFIGURATION - Modify these paths for your setup
# =============================================================================

class Config:
    # Paths - UPDATE THESE FOR YOUR SETUP
    checkpoint_dir = './checkpoints'  # Directory with G_epoch_X.pth and C_epoch_X.pth
    dataset_root = './cifar-10-batches-py'  # Or path to CelebA/other dataset
    output_dir = './analysis_results'

    # Analysis parameters
    start_epoch = 1
    end_epoch = 90                   # Adjust based on your checkpoints
    epoch_step = 1                    # Analyze every N epochs (increase for faster runs)

    # Image generation
    num_fake_images = 10000
    num_real_images = 50000
    batch_size = 32

    generator_feature_maps = 128
    critic_feature_maps = 128

    # Generator config (matched to checkpoint)
    latent_dim = 128
    image_size = 32                   # CIFAR-10 generator outputs 32x32
    image_channels = 3

    # Device - automatically selects CUDA > MPS > CPU
    device = DEVICE

    # DataLoader workers - use 0 for MPS to avoid multiprocessing issues
    num_workers = 0 if device.type == 'mps' else 4

config = Config()
os.makedirs(config.output_dir, exist_ok=True)

print(f"{'='*50}")
print(f"CONFIGURATION")
print(f"{'='*50}")
print(f"Device: {config.device}")
print(f"Checkpoints: {config.checkpoint_dir}")
print(f"Dataset: {config.dataset_root}")
print(f"Output: {config.output_dir}")
print(f"Epochs: {config.start_epoch} to {config.end_epoch} (step {config.epoch_step})")
print(f"Fake images per epoch: {config.num_fake_images}")
print(f"Real images for baseline: {config.num_real_images}")
print(f"Image size: {config.image_size}x{config.image_size}")
print(f"Batch size: {config.batch_size}")
print(f"{'='*50}")

CONFIGURATION
Device: cuda
Checkpoints: ./checkpoints
Dataset: ./cifar-10-batches-py
Output: ./analysis_results
Epochs: 1 to 90 (step 1)
Fake images per epoch: 5000
Real images for baseline: 10000
Image size: 32x32
Batch size: 32


In [ ]:
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.activations = {}
        self.main = nn.Sequential(
            nn.ConvTranspose2d(
                config.latent_dim,
                config.generator_feature_maps * 8,
                4,
                1,
                0,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 8),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 8,
                config.generator_feature_maps * 4,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 4,
                config.generator_feature_maps * 2,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps * 2,
                config.generator_feature_maps,
                4,
                2,
                1,
                bias=False,
            ),
            nn.BatchNorm2d(config.generator_feature_maps),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                config.generator_feature_maps,
                config.image_channels,
                3,
                1,
                1,
                bias=False,
            ),
            nn.Tanh(),
        )

    def forward(self, x):
        return self.main(x)



In [ ]:
class FastFeatureExtractor:
    """
    Extracts features
    """

    def __init__(self, device):
        self.device = device
        self.is_mps = (device.type == 'mps')

        # Load AlexNet
        print("Loading AlexNet...")
        self.alexnet = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        self.alexnet = self.alexnet.to(device).eval()
        print("✓ AlexNet loaded")

        self.conv_layers = ['conv1', 'conv2', 'conv3', 'conv4', 'conv5']
        self.fc_layers = ['fc1','fc2','fc3']
        self.layers = {
            'conv1': self.alexnet.features[0],
            'conv2': self.alexnet.features[3],
            'conv3': self.alexnet.features[6],
            'conv4': self.alexnet.features[8],
            'conv5': self.alexnet.features[10],
            'fc1': self.alexnet.classifier[1],
            'fc2': self.alexnet.classifier[4],
            'fc3': self.alexnet.classifier[6]
        }

        self.activations = {}
        self._register_hooks()

    def _register_hooks(self):
        def get_activation(name):
            def hook(model, input, output):
                self.activations[name] = output.detach()
            return hook
        for name, layer in self.layers.items():
            layer.register_forward_hook(get_activation(name))

    def preprocess(self, images, from_generator=False):
        """Preprocess images for AlexNet."""
        if from_generator:
            images = (images + 1) / 2  # [-1,1] -> [0,1]
            images = torch.clamp(images, 0, 1)
        images = F.interpolate(images, size=(224, 224), mode='bilinear', align_corners=False)
        mean = torch.tensor([0.485, 0.456, 0.406], device=images.device).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=images.device).view(1, 3, 1, 1)
        return (images - mean) / std

    def extract(self, images):
        """Extract features with layer-type-specific logic."""
        self.activations.clear()
        _ = self.alexnet(images)

        features = {}

        # ----------------------------
        # Convolutional layers
        # ----------------------------
        for layer in self.conv_layers:
            f = self.activations[layer]
            f = torch.flatten(f, start_dim=1)
            features[f"{layer}"] = f.cpu().numpy()

        # ----------------------------
        # Fully-connected layers
        # ----------------------------
        for layer in self.fc_layers:
            f = self.activations[layer]
            assert f.dim() == 2, f"FC layer not 2D: {layer}"
            features[f"{layer}"] = f.cpu().numpy()

        return features

extractor = FastFeatureExtractor(DEVICE)

Loading AlexNet...
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 230MB/s]


✓ AlexNet loaded


In [ ]:
# =============================================================================
# FID COMPUTATION + PCA (standardized: 2000 dims for all SPP layers)
# =============================================================================
from sklearn.decomposition import PCA

def compute_fid(feat1, feat2, regularized=False):
    """Compute FID between two feature sets."""
    mu1, mu2 = np.mean(feat1, axis=0), np.mean(feat2, axis=0)

    if regularized:
        sigma1 = LedoitWolf().fit(feat1).covariance_
        sigma2 = LedoitWolf().fit(feat2).covariance_
    else:
        sigma1 = np.cov(feat1, rowvar=False)
        sigma2 = np.cov(feat2, rowvar=False)

    diff = mu1 - mu2
    covmean = linalg.sqrtm(sigma1 @ sigma2)

    if np.iscomplexobj(covmean):
        covmean = covmean.real

    return float(diff @ diff + np.trace(sigma1 + sigma2 - 2 * covmean))

# PCA config - standardized 2000 dims for ALL SPP layers (matching Gram)
TARGET_DIM = 2000
feature_pca = {}
variance_explained = {}

def fit_feature_pca(real_features):
    global feature_pca
    print(f"Fitting PCA (threshold = {TARGET_DIM})")

    for key, feats in real_features.items():
        orig_dim = feats.shape[1]
        n_samples = feats.shape[0]

        if orig_dim > TARGET_DIM:
            n_components = min(TARGET_DIM, n_samples - 1)
            pca = PCA(n_components=n_components)
            pca.fit(feats)
            feature_pca[key] = pca
            variance_explained[key] = {
    "orig_dim": orig_dim,
    "pca_dim": n_components,
    "variance_explained": pca.explained_variance_ratio_.sum()
}

            print(
                f"  {key:<12}: {orig_dim} → {n_components} "
                f"(var {pca.explained_variance_ratio_.sum():.1%})"
            )
        else:
            print(f"  {key:<12}: {orig_dim} (no PCA)")

def apply_feature_pca(features, key):
    if key in feature_pca:
        return feature_pca[key].transform(features)
    return features

print(f"FID functions ready (SPP PCA: all layers -> {TARGET_DIM} dims)")

FID functions ready (SPP PCA: all layers -> 2000 dims)


In [ ]:
# =============================================================================
# DATA LOADING
# =============================================================================

from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms

def get_real_dataloader(dataset_root, batch_size=64, num_workers=0):
    # Standard AlexNet preprocessing
    transform = transforms.Compose([
        transforms.Resize(224),  # AlexNet expects 224x224
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    dataset_class = (
        torchvision.datasets.CIFAR10
    )
    dataset = dataset_class(
        root=config.dataset_root, train=True, transform=transform, download=True
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=(num_workers > 0),
        drop_last=False
    )

print("Data loading ready")

Data loading ready


In [ ]:
# =============================================================================
# EXTRACT REAL FEATURES
# =============================================================================

print("Extracting real image features...")

dataloader = get_real_dataloader(config.dataset_root,config.batch_size,num_workers=0)
real_features = None
count = 0

for images, _ in tqdm(dataloader, desc="Real images"):
    if count >= config.num_real_images:
        break


    images = images.to(DEVICE)

    batch_feat = extractor.extract(images)

    if real_features is None:
        real_features = {k: [v] for k, v in batch_feat.items()}
    else:
        for k, v in batch_feat.items():
            real_features[k].append(v)

    count += images.shape[0]

# Concatenate
for k in real_features:
    real_features[k] = np.concatenate(real_features[k], axis=0)[:config.num_real_images]


# Fit PCA for SPP features (skip small dims, reduce large dims to 1500)
print("\nFitting PCA for SPP features...")
fit_feature_pca(real_features)

df = (
    pd.DataFrame.from_dict(variance_explained, orient="index")
    .reset_index()
    .rename(columns={"index": "layer"})
)
df.to_csv(os.path.join(config.output_dir, 'variance_explained_PCA_cifar.csv'), index=False)



# Apply PCA to real SPP features
print("\nApplying PCA to real features...")
for key in list(real_features.keys()):
    real_features[key] = apply_feature_pca(real_features[key], key)
    print(f"  {key}: now {real_features[key].shape[1]} dims")

print("✓ Real features ready")

Extracting real image features...


100%|██████████| 170M/170M [00:03<00:00, 43.6MB/s]
Real images:  20%|██        | 313/1563 [00:24<01:38, 12.71it/s]



Fitting PCA for SPP features...
Fitting PCA (threshold = 2000)
  conv1       : 193600 → 2000 (var 93.6%)
  conv2       : 139968 → 2000 (var 84.1%)
  conv3       : 64896 → 2000 (var 77.3%)
  conv4       : 43264 → 2000 (var 76.1%)
  conv5       : 43264 → 2000 (var 80.7%)
  fc1         : 4096 → 2000 (var 97.6%)
  fc2         : 4096 → 2000 (var 98.7%)
  fc3         : 1000 (no PCA)

Applying PCA to real features...
  conv1: now 2000 dims
  conv2: now 2000 dims
  conv3: now 2000 dims
  conv4: now 2000 dims
  conv5: now 2000 dims
  fc1: now 2000 dims
  fc2: now 2000 dims
  fc3: now 1000 dims
✓ Real features ready


In [ ]:
print("Computing baselines...")

n_samples = real_features['conv1'].shape[0]
indices = np.random.permutation(n_samples)
split1, split2 = indices[:n_samples//2], indices[n_samples//2:]

baselines = {}

for key, features in tqdm(real_features.items(), desc="Baselines"):
    f1, f2 = features[split1], features[split2]

    # Standard FID baseline
    baselines[f"{key}_std"] = compute_fid(f1, f2, regularized=False)

Computing baselines...


Baselines: 100%|██████████| 8/8 [00:54<00:00,  6.84s/it]


In [ ]:
# =============================================================================
# MAIN ANALYSIS LOOP (optimized with PCA for SPP)
# =============================================================================
from google.colab import files

print("="*60)
print("ANALYZING EPOCHS")
print("="*60)

# Fixed noise for consistency
static_noise = torch.randn(config.num_fake_images, config.latent_dim, 1, 1, device=DEVICE)

results = []
epochs = list(range(config.start_epoch, config.end_epoch + 1, config.epoch_step))

for epoch in tqdm(epochs, desc="Epochs"):
    g_path = os.path.join(config.checkpoint_dir, f"G_epoch_{epoch}.pth")
    if not os.path.exists(g_path):
        continue

    # Load generator
    G = Generator().to(DEVICE)
    G.load_state_dict(torch.load(g_path, map_location=DEVICE, weights_only=True))
    G.eval()

    # Generate and extract features
    fake_features = None
    with torch.no_grad():
        for i in range(0, config.num_fake_images, config.batch_size):
            z = static_noise[i:i+config.batch_size]
            fake_imgs = G(z)
            fake_imgs = extractor.preprocess(fake_imgs, from_generator=True)
            batch_feat = extractor.extract(fake_imgs)

            if fake_features is None:
                fake_features = {k: [v] for k, v in batch_feat.items()}
            else:
                for k, v in batch_feat.items():
                    fake_features[k].append(v)

    for k in fake_features:
        fake_features[k] = np.concatenate(fake_features[k], axis=0)

    # Apply PCA to fake SPP features (using PCA fitted on real)
    for key in list(fake_features.keys()):
        fake_features[key] = apply_feature_pca(fake_features[key], key)

    # Compute FID for each method
    epoch_results = {'epoch': epoch}

    for layer in extractor.conv_layers:
        # SPP with standard FID (PCA-reduced where needed)
        fid_std = compute_fid(real_features[f"{layer}"], fake_features[f"{layer}"], regularized=False)
        epoch_results[f"{layer}_std"] = fid_std
        epoch_results[f"{layer}_std_norm"] = fid_std / baselines[f"{layer}_std"]

    for layer in extractor.fc_layers:
        fid_std = compute_fid((real_features[f"{layer}"]), fake_features[f"{layer}"], regularized = False)
        epoch_results[f"{layer}_std"] = fid_std
        epoch_results[f"{layer}_std_norm"] = fid_std / baselines[f"{layer}_std"]

    results.append(epoch_results)

    # Cleanup
    del G, fake_features
    gc.collect()
    if DEVICE.type == 'mps':
        torch.mps.empty_cache()

    # Progress update
    if len(results) % 1 == 0:
        print(f"  Completed {len(results)}/{len(epochs)} epochs")
        pd.DataFrame(results).to_csv(os.path.join(config.output_dir, 'results_partial.csv'), index=False)

# Save final
df = pd.DataFrame(results)
df.to_csv(os.path.join(config.output_dir, 'results_cifar.csv'), index=False)
print(f"\n✓ Saved results to {config.output_dir}/results_cifar.csv")

files.download(os.path.join(config.output_dir, 'results_cifar.csv'))
files.download(os.path.join(config.output_dir, 'results_partial.csv'))
files.download(os.path.join(config.output_dir, 'variance_explained_PCA_cifar.csv'))

ANALYZING EPOCHS


Epochs:   1%|          | 1/90 [01:21<2:01:15, 81.75s/it]

  Completed 1/90 epochs


Epochs:   2%|▏         | 2/90 [02:42<1:59:19, 81.36s/it]

  Completed 2/90 epochs


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>